In [14]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter
import numpy as np
from scipy.optimize import minimize


theta0 = Parameter('θ0')
theta1 = Parameter('θ1')
qc = QuantumCircuit(2, 2)
qc.ry(theta0, 0)
qc.ry(theta1, 1)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
print(qc.draw('text'))

sim = AerSimulator()
def cost_function(parameters):
    bound_circuit = qc.assign_parameters({theta0: parameters[0], theta1: parameters[1]})
    compiled_circuit = transpile(bound_circuit, sim)
    number = 2048
    result = sim.run(compiled_circuit, shots=number).result()
    counts = result.get_counts()

    p00 = counts.get('00', 0) / number
    p01 = counts.get('01', 0) / number
    p10 = counts.get('10', 0) / number
    p11 = counts.get('11', 0) / number
    cost = p01 + p10 + abs(p00 - p11)  # We want to minimize the probability of measuring |01> or |10>, and maximize the probability of measuring |00> or |11>
    return cost  # We want to minimize the probability of measuring |01> or |10>, and maximize the probability of measuring |00> or |11>

np.random.seed(42)
initial_guess = np.random.uniform(0, np.pi, 2)  # Random initial
print("Initial guess for θ0 and θ1:", initial_guess[0], initial_guess[1])
result = minimize(cost_function, initial_guess, method='COBYLA', options={'maxiter': 10000})

print("Optimized parameters for θ0 and θ1:", result.x[0], result.x[1])
print("Cost function value at optimized parameters:", result.fun)

#verifying the results
bound_circuit = qc.assign_parameters({theta0: result.x[0], theta1:result.x[1]})
compiled_circuit = transpile(bound_circuit, sim)
result = sim.run(compiled_circuit, shots=2048).result()
counts = result.get_counts()
print("Measurement counts at optimized parameters:", counts)


     ┌────────┐     ┌─┐   
q_0: ┤ Ry(θ0) ├──■──┤M├───
     ├────────┤┌─┴─┐└╥┘┌─┐
q_1: ┤ Ry(θ1) ├┤ X ├─╫─┤M├
     └────────┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 
Initial guess for θ0 and θ1: 1.176652485845522 2.9867570806801083
Optimized parameters for θ0 and θ1: 1.563285826532119 6.696799916071727
Cost function value at optimized parameters: 0.044921875
Measurement counts at optimized parameters: {'00': 979, '11': 981, '01': 36, '10': 52}
